# P08 — Rasa NLU con DIETClassifier
## (a) Paso del protocolo y objetivo
**Paso P08**: entrenar el clasificador **DIET** de Rasa NLU con la grilla fijada de antemano (`epochs ∈ {100,150,200}`, `batch_size ∈ {64,128}`, `embedding_dimension ∈ {20,50}`, semilla 42), **elegir la configuración en validación** y fijarla. Es el núcleo del **OE2** (diseñar e implementar la arquitectura del chatbot con Rasa NLU/DIET). La configuración elegida fue **100/64/20, semilla 42**, con un `FallbackClassifier` (umbral 0,50, ambigüedad 0,1; ver cuaderno 09).
**Origen:** resultados guardados sobre lenguaje **Real** (validación del lote 1) y la validación cruzada por participantes; el entrenamiento solo se reproduce si Rasa está instalado (en Colab **no**).

## (b) Código del repositorio que lo implementa
`run_rasa_grid.py` (grilla y repeticiones con `python -m rasa train nlu`), `eval_real.py` (selección en validación real y evaluación del test una vez), `crossval_agrupada.py` (validación cruzada por `base_phrase_id`) y `crossval_participantes.py` (dejando participantes fuera). Entradas: `data/*.yml`, `configs/rasa_config*.yml`. Salidas: `logs/v3_real/RASA-e*-b*-d*-s*/` (métricas y predicciones; **privadas**), `logs/v3_real/rasa_validation.csv` (números).

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("08"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,crossval_agrupada.py,P11,P11 — Validación cruzada AGRUPADA por base_phrase_id (sin fuga por paráfrasis).,Corpus sintético,Validación cruzada agrupada por base_phrase_id,python scripts/crossval_agrupada.py # baseline + Rasa (Rasa tarda ~30 min),Sí (python -m rasa),Documentado (requiere Rasa)
1,crossval_participantes.py,Sensibilidad (V1.6),ANÁLISIS DE SENSIBILIDAD — validación cruzada dejando participantes fuera (protocolo V1.2).,Frases reales del lote 1 por participante,logs/avance/cv_participantes_*.md (no cuenta para G3),python scripts/crossval_participantes.py --help,No,Documentado (requiere Rasa)
2,demo_vivo.py,"Demostración en vivo (P07, P08)",Piezas en Python de la demostración en vivo (las orquesta scripts/demo_vivo.ps1; se pueden correr sueltas).,"Corpus sintético, configs/","models/demo_vivo/ (SVM y DIET de demostración), huellas de archivos protegidos, tabla de pruebas",python scripts/demo_vivo.py snapshot guardar|comparar huellas de los archivos protegidos (antes y después de la demo),No,Documentado (se ejecuta con demo_vivo.ps1; requiere Rasa)
3,entrenar_modelo_demo.py,Demostración (P08),"Entrena el modelo de DEMOSTRACIÓN del asistente: Rasa/DIET con la configuración del modelo congelado, pero SOLO con el corpus SINTÉTICO (data/nlu_train.yml, 546 frases).",data/nlu_train.yml (sintético) + configs/rasa_config_lote2.yml (copia),models/demo_vivo/DEMO-VIVO.tar.gz,python scripts/entrenar_modelo_demo.py # configuración oficial: 100 épocas (unos 5 min),Sí (python -m rasa),Documentado (requiere Rasa; solo datos sintéticos)
4,eval_real.py,"P08, P10, P11 (lote 1)","Evaluación sobre lenguaje real (protocolo V1.2, P08/P10/P11) — partición v3.",Partición v3 real,"logs/v3_real/ (F1 macro, selección por validación, test UNA vez)",python scripts/eval_real.py # selección + test (Rasa tarda ~1 h 40 min),Sí (import),Documentado (el test del lote 1 ya se gastó)
5,run_rasa_grid.py,"P08, P10, P11","P08 / P10 / P11 — Rasa NLU con DIETClassifier: grilla, selección y repeticiones.",data/*.yml + configs/rasa_config.yml,logs/RASA-e<epochs>-b<batch>-d<dim>-s<semilla>/,python scripts/run_rasa_grid.py # experimento completo (12 + 5 entrenamientos),Sí (import),Documentado (requiere Rasa)


## (c) Celdas de código
### 1. Configuración congelada de DIET y del FallbackClassifier

In [3]:
import yaml
cfg = yaml.safe_load((BASE / "configs/rasa_config_lote2.yml").read_text(encoding="utf-8"))
bcfg = leer_json("configs/baseline_config.json")
diet = next(c for c in cfg["pipeline"] if c["name"] == "DIETClassifier")
fb = next(c for c in cfg["pipeline"] if c["name"] == "FallbackClassifier")
rotulo(ORIGEN_SINTETICO, "configuración (código, no datos)")
display(pd.DataFrame([
    {"modelo": "DIET (congelado)", "parámetro": k, "valor": diet[k]} for k in ("epochs", "batch_size", "embedding_dimension", "random_seed", "learning_rate")] + [
    {"modelo": "FallbackClassifier", "parámetro": "threshold", "valor": fb["threshold"]}, {"modelo": "FallbackClassifier", "parámetro": "ambiguity_threshold", "valor": fb["ambiguity_threshold"]}] + [
    {"modelo": "SVM baseline", "parámetro": "vectorizador", "valor": f"TF-IDF max_features={bcfg['vectorizer']['max_features']}, ngram_range={tuple(bcfg['vectorizer']['ngram_range'])}"},
    {"modelo": "SVM baseline", "parámetro": "clasificador", "valor": f"SVC kernel={bcfg['classifier']['kernel']}, C ∈ {bcfg['classifier']['C_grid']}"},
    {"modelo": "SVM baseline", "parámetro": "criterio de selección", "valor": bcfg["classifier"]["selection_criterion"]}]))
comparar("P07/P08", "DIET epochs/batch/dim/semilla", f"{diet['epochs']}/{diet['batch_size']}/{diet['embedding_dimension']}/{diet['random_seed']}", "100/64/20/42", "configs/rasa_config_lote2.yml", tipo="txt")
comparar("P07/P08", "umbral t y ambigüedad", f"{fb['threshold']}/{fb['ambiguity_threshold']}", "0.5/0.1", "configs/rasa_config_lote2.yml", tipo="txt")

▌ORIGEN: Sintético — configuración (código, no datos)


,modelo,parámetro,valor
0,DIET (congelado),epochs,100
1,DIET (congelado),batch_size,64
2,DIET (congelado),embedding_dimension,20
3,DIET (congelado),random_seed,42
4,DIET (congelado),learning_rate,0.001
5,FallbackClassifier,threshold,0.5
6,FallbackClassifier,ambiguity_threshold,0.1
7,SVM baseline,vectorizador,"TF-IDF max_features=5000, ngram_range=(1, 2)"
8,SVM baseline,clasificador,"SVC kernel=linear, C ∈ [0.1, 1, 10]"
9,SVM baseline,criterio de selección,mejor F1 macro en validación


### 2. La grilla en validación real (Real, cifras guardadas)
Se lee `logs/v3_real/rasa_validation.csv` (una fila por configuración, semilla 42) y se comprueba que la configuración congelada es la de mayor F1 macro en validación, como declara `seleccion_final.json`.

In [4]:
val = pd.read_csv(BASE / "logs/v3_real/rasa_validation.csv")
sel = leer_json("logs/v3_real/seleccion_final.json")
val = val.sort_values("f1_macro", ascending=False).reset_index(drop=True)
rotulo(ORIGEN_REAL, "F1 macro en la validación real del lote 1 (71 frases) por configuración de DIET")
display(val[["experiment_id", "epochs", "batch_size", "embedding_dimension", "f1_macro", "accuracy", "train_seconds"]].round(4))
mejor = val.iloc[0]
print("Configuración de mayor F1 en validación:", mejor["experiment_id"], f"(F1 {mejor['f1_macro']:.4f})")
comparar("P08", "configuraciones de la grilla", len(val), 12, "Protocolo: 3 × 2 × 2", tol=0)
comparar("P08", "configuración elegida en validación", mejor["experiment_id"], sel["rasa"]["exp_id"], "logs/v3_real/seleccion_final.json", tipo="txt")
comparar("P08", "F1 macro de la configuración elegida (validación)", mejor["f1_macro"], sel["rasa"]["f1_macro_validacion"], "logs/v3_real/seleccion_final.json", tol=1e-9)
dos = val.head(2)
print("La segunda mejor configuración queda a", round(float(dos.iloc[0]["f1_macro"] - dos.iloc[1]["f1_macro"]), 4), "de la primera: con 71 frases la elección es poco firme.")

▌ORIGEN: Real — F1 macro en la validación real del lote 1 (71 frases) por configuración de DIET


,experiment_id,epochs,batch_size,embedding_dimension,f1_macro,accuracy,train_seconds
0,RASA-e100-b64-d20-s42,100,64,20,0.6809,0.7042,306.7
1,RASA-e200-b64-d20-s42,200,64,20,0.6802,0.7042,766.2
2,RASA-e200-b128-d20-s42,200,128,20,0.6580,0.7042,706.6
3,RASA-e150-b64-d20-s42,150,64,20,0.6525,0.6901,523.1
4,RASA-e150-b128-d20-s42,150,128,20,0.6506,0.6901,532.6
5,RASA-e100-b64-d50-s42,100,64,50,0.6414,0.6761,279.7
6,RASA-e100-b128-d50-s42,100,128,50,0.6327,0.6761,259.7
7,RASA-e150-b128-d50-s42,150,128,50,0.6080,0.6479,531.4
8,RASA-e200-b128-d50-s42,200,128,50,0.6080,0.6479,644.9
9,RASA-e150-b64-d50-s42,150,64,50,0.5858,0.6338,599.9


Configuración de mayor F1 en validación: RASA-e100-b64-d20-s42 (F1 0.6809)
La segunda mejor configuración queda a 0.0006 de la primera: con 71 frases la elección es poco firme.


### 3. Validación cruzada dejando participantes fuera (Real, análisis de sensibilidad)
No cuenta para G3 ni para decidir ajustes. Compara DIET y SVM con las 185 frases reales activas del lote 1 (31 participantes, 5 folds por participante).

In [5]:
cvp = (BASE / "logs/avance/cv_participantes_resumen.md").read_text(encoding="utf-8").splitlines()
rotulo(ORIGEN_REAL, "texto guardado de logs/avance/cv_participantes_resumen.md (solo cifras agregadas)")
modelo = None
for l in cvp:
    if l.startswith("## "):
        modelo = l[3:].strip()
    elif l.startswith("- F1 macro global"):
        print(f"{modelo}: {l[2:240]}")
        if modelo == "RASA":
            m = re.search(r"\*\*([\d.]+)\*\*", l)
            comparar("P08", "F1 macro global de la CV por participantes (DIET)", float(m.group(1)), 0.7866, "cv_participantes_resumen.md", tol=5e-4)

▌ORIGEN: Real — texto guardado de logs/avance/cv_participantes_resumen.md (solo cifras agregadas)
SVM: F1 macro global (54 intenciones, predicciones de todos los folds juntas): **0.7816**, IC95 % por bootstrap de participantes [0.6414, 0.7995]; accuracy 0.7784.
RASA: F1 macro global (54 intenciones, predicciones de todos los folds juntas): **0.7866**, IC95 % por bootstrap de participantes [0.6530, 0.8022]; accuracy 0.7892.


### 4. Entrenamiento de DIET (opcional, solo con Rasa)
En Colab esta celda no entrena nada. Con Rasa 3.6 / Python 3.10 y `MPSR_ENTRENAR_DIET=1`, entrena una **demostración Sintética** en carpeta aparte; no reemplaza al modelo congelado.

In [6]:
cm = pd.read_csv(BASE / "corpus/corpus_metadata.csv", dtype=str, keep_default_na=False, encoding="utf-8")
DEMO_DIR = BASE / "demo_entrenamiento"
DEMO_DIR.mkdir(exist_ok=True)

In [7]:
# ----- Entrenamiento DIET opcional (solo con Rasa) -----
if HAY_RASA and os.environ.get("MPSR_ENTRENAR_DIET") == "1":
    import subprocess
    salida_rasa = DEMO_DIR / "rasa_demo"
    cmd = [sys.executable, "-m", "rasa", "train", "nlu", "--nlu", str(BASE / "data/nlu_train.yml"), "--config", str(BASE / "configs/rasa_config.yml"),
           "--out", str(salida_rasa), "--fixed-model-name", "DEMO-SINTETICO"]
    print("DEMOSTRACIÓN (Sintético) — no sustituye al modelo congelado:", " ".join(cmd))
    subprocess.run(cmd, check=False)
else:
    print("Entrenamiento DIET omitido: " + ("Rasa no está instalado aquí (lo esperado en Colab)." if not HAY_RASA else "para activarlo define MPSR_ENTRENAR_DIET=1."))
    print("El resultado del entrenamiento real de DIET está en los cuadernos 12 (congelamiento) y 10 (predicciones guardadas).")

Entrenamiento DIET omitido: Rasa no está instalado aquí (lo esperado en Colab).
El resultado del entrenamiento real de DIET está en los cuadernos 12 (congelamiento) y 10 (predicciones guardadas).


In [8]:
cierre("P08 Rasa/DIET")

P08 Rasa/DIET — recalculado frente a reportado: 6 cifras · coinciden: 6 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P07/P08,DIET epochs/batch/dim/semilla,100/64/20/42,100/64/20/42,configs/rasa_config_lote2.yml,Sí
1,P07/P08,umbral t y ambigüedad,0.5/0.1,0.5/0.1,configs/rasa_config_lote2.yml,Sí
2,P08,configuraciones de la grilla,12,12,Protocolo: 3 × 2 × 2,Sí
3,P08,configuración elegida en validación,RASA-e100-b64-d20-s42,RASA-e100-b64-d20-s42,logs/v3_real/seleccion_final.json,Sí
4,P08,F1 macro de la configuración elegida (validación),0.680864,0.680864,logs/v3_real/seleccion_final.json,Sí
5,P08,F1 macro global de la CV por participantes (DIET),0.7866,0.7866,cv_participantes_resumen.md,Sí


DISCREPANCIAS (recalculado ≠ reportado): ninguna
Rasa disponible: False · paquete privado: True


## (d) Cómo leer el resultado
- La **tabla de la grilla** muestra que 100/64/20 fue la mejor en validación (F1 0,681), seguida de cerca por 200/64/20 (0,680). Con 71 frases de validación esa diferencia es ruido: la configuración se fijó **antes** de mirar el test y no se volvió a tocar (salvo registro en la bitácora).
- La **validación cruzada por participantes** da F1 ≈ 0,787 para DIET y 0,782 para el SVM: las dos familias rinden parecido con 185 frases reales.
- Entrenar DIET exige Rasa; por eso aquí se muestran resultados guardados y las huellas del modelo (cuaderno 12).

## Limitaciones
- La grilla se eligió con una validación de **71 frases** (lote 1): el IC95 % del F1 de validación (0,54–0,74) es muy ancho.
- La CV por participantes incluye frases que luego formaron parte del test del lote 1; por eso es solo un análisis de sensibilidad.